# OneHotEncoder

## Example 1

In [24]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OneHotEncoder, OrdinalEncoder

from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# avoid warnings
import warnings
warnings.filterwarnings('ignore')

In [16]:

# Create sample dataset
data = pd.DataFrame({
    'city': ['NYC', 'LA', 'SF', 'NYC', 'LA', 'SF', 'NYC', 'Boston'],
    'color': ['red', 'blue', 'green', 'red', 'blue', 'red', 'green', 'red'],
    'size': [10, 20, 15, 12, 18, 14, 16, 11],
    'target': [1, 0, 1, 1, 0, 1, 0, 1]
})

print("Original Data:")
print(data.head())

Original Data:
  city  color  size  target
0  NYC    red    10       1
1   LA   blue    20       0
2   SF  green    15       1
3  NYC    red    12       1
4   LA   blue    18       0


In [18]:
# Split data
X = data.drop('target', axis=1)
y = data['target']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

### Method 1: Using OneHotEncoder directly

In [19]:
print("\n" + "="*60)
print("Method 1: Basic OneHotEncoder")
print("="*60)

encoder = OneHotEncoder(sparse_output=False, drop='first')  # drop='first' avoids dummy trap
encoded_train = encoder.fit_transform(X_train[['city', 'color']])

print(f"\nOriginal features: {encoder.feature_names_in_}")
print(f"Encoded features: {encoder.get_feature_names_out()}")
print(f"\nEncoded data shape: {encoded_train.shape}")
print(f"First few rows:\n{encoded_train[:3]}")


Method 1: Basic OneHotEncoder

Original features: ['city' 'color']
Encoded features: ['city_LA' 'city_NYC' 'city_SF' 'color_green' 'color_red']

Encoded data shape: (6, 5)
First few rows:
[[0. 1. 0. 0. 1.]
 [0. 0. 0. 0. 1.]
 [0. 0. 1. 1. 0.]]


### Method 2: Using ColumnTransformer (Recommended)

In [20]:
print("\n" + "="*60)
print("Method 2: ColumnTransformer with Pipeline")
print("="*60)

# Define preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), 
         ['city', 'color']),
        ('num', 'passthrough', ['size'])
    ]
)

# Create full pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42))
])

# Fit and predict
pipeline.fit(X_train, y_train)
train_score = pipeline.score(X_train, y_train)
test_score = pipeline.score(X_test, y_test)

print(f"\nTraining accuracy: {train_score:.3f}")
print(f"Testing accuracy: {test_score:.3f}")


Method 2: ColumnTransformer with Pipeline

Training accuracy: 1.000
Testing accuracy: 1.000


### Method 3: Handling unseen categories

In [21]:
print("\n" + "="*60)
print("Method 3: Handling Unseen Categories")
print("="*60)

# Create test data with unseen category
X_new = pd.DataFrame({
    'city': ['Chicago'],  # Unseen category
    'color': ['red'],
    'size': [15]
})

print("\nNew data with unseen city 'Chicago':")
print(X_new)

try:
    prediction = pipeline.predict(X_new)
    print(f"Prediction (with handle_unknown='ignore'): {prediction}")
    print("Success! The unseen category was handled gracefully.")
except Exception as e:
    print(f"Error: {e}")


Method 3: Handling Unseen Categories

New data with unseen city 'Chicago':
      city color  size
0  Chicago   red    15
Prediction (with handle_unknown='ignore'): [1]
Success! The unseen category was handled gracefully.


In [22]:
# Method 4: pandas get_dummies (simple but less flexible)
print("\n" + "="*60)
print("Method 4: pandas get_dummies (Quick Exploration)")
print("="*60)

dummies = pd.get_dummies(
    data[['city', 'color']], 
    drop_first=True,  # Avoid dummy trap
    prefix=['city', 'color']
)
print("\nOne-hot encoded with pandas:")
print(dummies.head())


Method 4: pandas get_dummies (Quick Exploration)

One-hot encoded with pandas:
   city_LA  city_NYC  city_SF  color_green  color_red
0    False      True    False        False       True
1     True     False    False        False      False
2    False     False     True         True      False
3    False      True    False        False       True
4     True     False    False        False      False


In [23]:
# Method 5: Demonstrating sparse vs dense
print("\n" + "="*60)
print("Method 5: Sparse vs Dense Output")
print("="*60)

encoder_sparse = OneHotEncoder(sparse_output=True)
encoder_dense = OneHotEncoder(sparse_output=False)

sparse_output = encoder_sparse.fit_transform(X_train[['city']])
dense_output = encoder_dense.fit_transform(X_train[['city']])

print(f"\nSparse matrix type: {type(sparse_output)}")
print(f"Sparse matrix shape: {sparse_output}")
print(f"Sparse matrix memory (bytes): {sparse_output.data.nbytes}")

print(f"\nDense array type: {type(dense_output)}")
print(f"Dense array shape: {dense_output}")
print(f"Dense array memory (bytes): {dense_output.nbytes}")


Method 5: Sparse vs Dense Output

Sparse matrix type: <class 'scipy.sparse._csr.csr_matrix'>
Sparse matrix shape: <Compressed Sparse Row sparse matrix of dtype 'float64'
	with 6 stored elements and shape (6, 4)>
  Coords	Values
  (0, 2)	1.0
  (1, 0)	1.0
  (2, 3)	1.0
  (3, 1)	1.0
  (4, 2)	1.0
  (5, 2)	1.0
Sparse matrix memory (bytes): 48

Dense array type: <class 'numpy.ndarray'>
Dense array shape: [[0. 0. 1. 0.]
 [1. 0. 0. 0.]
 [0. 0. 0. 1.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]]
Dense array memory (bytes): 192


## Advanced Example: Comparing Encodings

In [25]:
# Generate dataset with different cardinalities
np.random.seed(42)
n_samples = 1000

data = pd.DataFrame({
    'low_card': np.random.choice(['A', 'B', 'C'], n_samples),  # 3 categories
    'med_card': np.random.choice(list('ABCDEFGHIJ'), n_samples),  # 10 categories
    'high_card': np.random.choice(range(100), n_samples),  # 100 categories
    'numeric': np.random.randn(n_samples)
})

# Create target with some relationship to features
data['target'] = (
    (data['low_card'] == 'A').astype(int) +
    (data['med_card'].isin(['A', 'B'])).astype(int) +
    np.random.randint(0, 2, n_samples)
) % 2

X = data.drop('target', axis=1)
y = data['target']


### 1. LogisticRegression with Mixed Cardinality Features

In [26]:
print("\n" + "="*60)
print("LogisticRegression with Mixed Cardinality Features")
print("="*60)

pipeline = Pipeline([
    ('encoder', ColumnTransformer([
        ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore'), ['low_card', 'med_card']),
        ('ordinal', OrdinalEncoder(), ['high_card']),
        ('num', 'passthrough', ['numeric'])
    ])),
    ('clf', LogisticRegression(max_iter=1000))
])

scores = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy')
print(f"  Mean CV Score: {scores.mean():.4f} (+/- {scores.std():.4f})")


LogisticRegression with Mixed Cardinality Features
  Mean CV Score: 0.5190 (+/- 0.0235)


In [27]:
print("\n" + "="*60)
print("LogisticRegression with Mixed Cardinality Features")
print("="*60)

pipeline = Pipeline([
    ('encoder', ColumnTransformer([
        ('ordinal', OrdinalEncoder(), ['low_card', 'med_card', 'high_card']),
        ('num', 'passthrough', ['numeric'])
    ])),
    ('clf', LogisticRegression(max_iter=1000))
])

scores = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy')
print(f"  Mean CV Score: {scores.mean():.4f} (+/- {scores.std():.4f})")


LogisticRegression with Mixed Cardinality Features
  Mean CV Score: 0.5220 (+/- 0.0211)


### RandomForest

In [28]:
print("\n" + "="*60)
print("RandomForest with Mixed Cardinality Features")
print("="*60)

pipeline = Pipeline([
    ('encoder', ColumnTransformer([
        ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore'), ['low_card', 'med_card', 'high_card']),
        ('num', 'passthrough', ['numeric'])
    ])),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
])

scores = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy')
print(f"  Mean CV Score: {scores.mean():.4f} (+/- {scores.std():.4f})")


RandomForest with Mixed Cardinality Features
  Mean CV Score: 0.5210 (+/- 0.0314)


In [29]:
print("\n" + "="*60)
print("RandomForest with Mixed Cardinality Features")
print("="*60)

pipeline = Pipeline([
    ('encoder', ColumnTransformer([
        ('ordinal', OrdinalEncoder(), ['low_card', 'med_card', 'high_card']),
        ('num', 'passthrough', ['numeric'])])),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
])

scores = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy')
print(f"  Mean CV Score: {scores.mean():.4f} (+/- {scores.std():.4f})")


RandomForest with Mixed Cardinality Features
  Mean CV Score: 0.5170 (+/- 0.0220)
